<a href="https://colab.research.google.com/github/sp0ntanius/tcc_sumarizacao_tfidf/blob/main/Testes_extrativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rouge_score bert_score spacy pandas scikit-learn datasets
!python -m spacy download pt_core_news_lg


In [7]:
# IMPORTAÇÃO DA BASE DE DADOS

from datasets import load_dataset
import pandas as pd

# Conjunto de dados RecognaSumm
conj_dados = load_dataset("recogna-nlp/recognasumm")

# Converter a parte de treinamento para uma tabela para facilitar a visualização
table_treino = pd.DataFrame(conj_dados['train'])

# print das 5 primeiras linhas da tabela
table_treino.head()

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=e9d8d89e8d4e9d3db6d9f5195af6e0dce364b9addf3cfbba738329755e4e4124
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


,index,Titulo,Subtitulo,Noticia,Categoria,Autor,Data,URL,Autor_corrigido,Sumario
0,52037,Ex-BBB Paula se emociona com novo procedimento...,Ex-sister compartilhou vídeo da preparação par...,A ex-BBB 23 Paula Freitas compartilhou em suas...,Entretenimento,Extra,09/05/2023 10h53,https://extra.globo.com/entretenimento/noticia...,Extra,Ex-BBB Paula se emociona com novo procedimento...
1,45191,Trump sinaliza que anúncio de candidatura par...,"Em entrevista à New York Magazine, ex-presiden...",Gabby Orrda CNN Talvez no sinal mais forte até...,Internacional,CNN,14/07/2022 às 16:51 | Atualizado 14/07/2022 à...,https://www.cnnbrasil.com.br/internacional/tru...,CNN,Trump sinaliza que anúncio de candidatura para...
2,74910,Auxílio Emergencial 2021: Caixa começa a pagar...,Pagamento começa nesta quinta para trabalhador...,A Caixa Econômica Federal (Caixa) começa a pag...,Economia,Por G1,17/06/2021 00h00,https://g1.globo.com/economia/auxilio-emergenc...,G1,Auxílio Emergencial 2021: Caixa começa a pagar...
3,120941,Netflix lança novos recursos para controle dos...,Plataforma vai permitir que usuários coloquem ...,A empresa também adicionou um recurso para pro...,Política,Por Reuters,07/04/2020 22h02,https://g1.globo.com/economia/tecnologia/notic...,Reuters,Netflix lança novos recursos para controle dos...
4,114965,Estudo estima que 5% da população da Espanha c...,"Na região da capital, 11% da população teve te...","""O estudo constata que 5% da população espanho...",Ciência e Tecnologia,Por France Presse,13/05/2020 21h12,https://g1.globo.com/mundo/noticia/2020/05/13/...,France Presse,Estudo estima que 5% da população da Espanha c...


In [8]:
# UTILIZAÇÃO DA TABELA DE TREINO

# Listagem das colunas
print("Colunas do conjunto de dados:")
print(table_treino.columns.tolist())
print("-" * 50)

# Exibição da primeira notícia na íntegra
not_exemplo = table_treino ['Noticia'][0]
print("Conteúdo da primeira notícia:")
print(not_exemplo)

# Exibição do sumário da primeira notícia
sum_exemplo = table_treino['Sumario'][0]
print("\nSumário da primeira notícia:")
print(sum_exemplo)

Colunas do conjunto de dados:
['index', 'Titulo', 'Subtitulo', 'Noticia', 'Categoria', 'Autor', 'Data', 'URL', 'Autor_corrigido', 'Sumario']
--------------------------------------------------
Conteúdo da primeira notícia:
A ex-BBB 23 Paula Freitas compartilhou em suas redes sociais um vídeo dos bastidores de um procedimento estético que realizou no corpo. A biomédica, que já havia realizado procedimentos no rosto, como preenchimento labial, se emocionou ao mostrar a novidade e contou que tem esse sonho desde a adolescência: "Desde os 16 anos que eu tenho o sonho de fazer isso. E às vezes eu ficava me questionando se foi muita coisa que eu ouvi na vida... Eu espero que realmente saia da forma como eu estou imaginando porque eu idealizo isso desde muito nova. Quem sabe, já pensou eu vou parar na Sapucaí sambando? Aí é loucura, né?!" No vídeo, Paula mostra as marcações no corpo, presentes em áreas como a região dos seios e da barriga. A ex-sister foi para o hospital onde realizou a cirurg

In [12]:
import spacy

# Baixar e carregar a base de conhecimento gramatical em português
nlp = spacy.load("pt_core_news_lg")

# Função de limpeza do texto
def pre_proc_text(texto):
    if not isinstance(texto, str):    # se o texto for nulo ou inválido, retorna vazio
        return ""

    # processamento de texto (convertendo para minúsculas)
    documento = nlp(texto.lower())

    tokens_limpos = []
    for palavra in documento: # filtro de palavras vazias, pontuações e espaço em branco
        if not palavra.is_stop and not palavra.is_punct and not palavra.is_space:
            tokens_limpos.append(palavra.lemma_)

    return " ".join(tokens_limpos) # retorno das palavras limpas em um texto

# teste de função com a primeira notícia
not_limpa = pre_proc_text(not_exemplo)

print("Texto original (500 primeiros caracteres):")
print(not_exemplo[:500] + "...\n")

print("Texto após processamento (500 primeiros caracteres):")
print(not_limpa[:500] + "...")

Texto original (500 primeiros caracteres):
A ex-BBB 23 Paula Freitas compartilhou em suas redes sociais um vídeo dos bastidores de um procedimento estético que realizou no corpo. A biomédica, que já havia realizado procedimentos no rosto, como preenchimento labial, se emocionou ao mostrar a novidade e contou que tem esse sonho desde a adolescência: "Desde os 16 anos que eu tenho o sonho de fazer isso. E às vezes eu ficava me questionando se foi muita coisa que eu ouvi na vida... Eu espero que realmente saia da forma como eu estou imagina...

Texto após processamento (500 primeiros caracteres):
ex-bbb 23 Paula freitas compartilhar rede social vídeo bastidor procedimento estético realizar corpo biomédica haver realizar procedimento rosto preenchimento labial emocionar mostrar novidade contar sonho adolescência 16 ano sonho ficar questionar muito ouvir vida esperar realmente saia imaginar idealizo pensar ir parar sapucaí sambar loucura né vídeo paula mostrar marcação corpo presente área r

In [11]:
# TESTE DE CÁLUCLO DO TF-IDF I
from sklearn.feature_extraction.text import TfidfVectorizer

# dividindo a notícia original em frases matemáticas
doc_spacy = nlp(not_exemplo)
frases_orig = [sentenca.text for sentenca in doc_spacy.sents]

# limpar cada frase individualmente
frases_limp = [pre_proc_text(frase) for frase in frases_orig]

# inicializar calculadora estatística
vetorizador = TfidfVectorizer()

# construir matriz matemática com o peso de cada palavra nas frases
matriz_tfidf = vetorizador.fit_transform(frases_limp)

# resultados
print(f"A notícia foi dividida em {len(frases_orig)} frases distintas.")
print(f"Após a limpeza, o algoritmo identificou {len(vetorizador.get_feature_names_out())} palavras únicas essenciais.")

A notícia foi dividida em 14 frases distintas.
Após a limpeza, o algoritmo identificou 69 palavras únicas essenciais.


In [13]:
# TESTE DE CÁLCULO DO TF-IDF II

import numpy as np

# lista das pontuações das frases (pontuação 0, frase 0)
pont_frases = np.array(matriz_tfidf.sum(axis=1)).flatten()

# associar pontuação e a posição original com cada frase
frases_com_pont = []
for indice, pont in enumerate(pont_frases):
    frases_com_pont.append((indice, pont, frases_orig[indice]))

# ordenar frases da maior pontuação para a menor
frases_ord = sorted(frases_com_pont, key=lambda x: x[1], reverse=True)

# definir o tamanho do resumo e selecioná-las
qtdd_frases_resum = 3
melhores_frases = frases_ord[:qtdd_frases_resum]

# reordenar as frases escolhidas para a ordem cronológica que aparecem no texto
melhores_frases_cron = sorted(melhores_frases, key=lambda x:x[0])

# unir frases para o texto final
resumo_gerado = " ".join([frase[2] for frase in melhores_frases_cron])

# resultados
print("Resumo gerado pelo algoritmo:\n", resumo_gerado)
print("-" * 50)
print("Resumo de referência (gabarito humano):\n", table_treino['Sumario'][0])


Resumo gerado pelo algoritmo:
 Paula Freitas compartilhou em suas redes sociais um vídeo dos bastidores de um procedimento estético que realizou no corpo. A biomédica, que já havia realizado procedimentos no rosto, como preenchimento labial, se emocionou ao mostrar a novidade e contou que tem esse sonho desde a adolescência: No vídeo, Paula mostra as marcações no corpo, presentes em áreas como a região dos seios e da barriga.
--------------------------------------------------
Resumo de referência (gabarito humano):
 Ex-BBB Paula se emociona com novo procedimento estético: 'Desde os 16 anos que sonho em fazer isso' Ex-sister compartilhou vídeo da preparação para a cirurgia no corpo.


In [14]:
# EXIBIÇÃO DA MÉTRICA ROUGE PARA AVALIAÇÃO DO ALGORITMO SELECIONADO

from rouge_score import rouge_scorer

# isolamento do resumo de referência
resumo_ref = table_treino['Sumario'][0]

# inicialização do calculador Rouge
avaliador = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

# calculando pontuação do nosso resumo com a referência
pontuacao = avaliador.score(resumo_ref, resumo_gerado)

# resultados
print("Resultados da avaliação Rouge:\n")

for metrica, valores in pontuacao.items():
    print(f"--- {metrica.upper()} ---")
    print(f"Precision: {valores.precision * 100:.2f}%")
    print(f"Recall  : {valores.recall * 100:.2f}%")
    print(f"F1-Score: {valores.fmeasure * 100:.2f}%\n")

Resultados da avaliação Rouge:

--- ROUGE1 ---
Precision: 23.61%
Recall  : 53.12%
F1-Score: 32.69%

--- ROUGE2 ---
Precision: 5.63%
Recall  : 12.90%
F1-Score: 7.84%

--- ROUGEL ---
Precision: 13.89%
Recall  : 31.25%
F1-Score: 19.23%



In [18]:
# EXIBIÇÃO DA MÉTRICA BERTSCORE PARA AVALIAÇÃO SEMÂNTICA DO EXEMPLO ISOLADO

import evaluate

print("Carregando o modelo BERTScore...")
avaliador_bertscore = evaluate.load("bertscore")

# Calculando a pontuação semântica do nosso resumo com a referência
# IMPORTANTE: O BERTScore exige que as variáveis estejam dentro de listas [ ], mesmo sendo apenas um texto.
pontuacao_bert = avaliador_bertscore.compute(
    predictions=[resumo_gerado],
    references=[resumo_ref],
    lang="pt"
)

# Resultados
print("\nResultados da avaliação BERTScore:\n")
print("--- BERTSCORE (Análise Semântica) ---")
print(f"Precision: {pontuacao_bert['precision'][0] * 100:.2f}%")
print(f"Recall   : {pontuacao_bert['recall'][0] * 100:.2f}%")
print(f"F1-Score : {pontuacao_bert['f1'][0] * 100:.2f}%\n")

Carregando o modelo BERTScore (isso pode levar alguns segundos na primeira vez)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Resultados da avaliação BERTScore:

--- BERTSCORE (Análise Semântica) ---
Precision: 65.98%
Recall   : 73.39%
F1-Score : 69.49%



In [15]:
# TESTE DE MUDANÇA DE FRASES SELECIONADAS NO ALGORITMO TF-IDF (ROUGE)

# redefinição do tamanho do resumo para apenas 2 frases
qtdd_frases_resumo = 2
melhores_frases = frases_ord[:qtdd_frases_resumo]

# reordenar cronologicamente e unir o texto
melhores_frases_cron = sorted(melhores_frases, key=lambda x: x[0])
resumo_gerado_2_frases = " ".join([frase[2] for frase in melhores_frases_cron])

print("Resumo gerado (2 Frases):\n", resumo_gerado_2_frases)
print("-" * 50)

# recalcular a pontuação ROUGE com o novo resumo
pontuacao_2_frases = avaliador.score(resumo_ref, resumo_gerado_2_frases)

print("Novos resultados da avaliação Rouge:\n")
for metrica, valores in pontuacao_2_frases.items():
    print(f"--- {metrica.upper()} ---")
    print(f"Precision: {valores.precision * 100:.2f}%")
    print(f"Recall   : {valores.recall * 100:.2f}%")
    print(f"F1-Score : {valores.fmeasure * 100:.2f}%\n")

Resumo gerado (2 Frases):
 Paula Freitas compartilhou em suas redes sociais um vídeo dos bastidores de um procedimento estético que realizou no corpo. A biomédica, que já havia realizado procedimentos no rosto, como preenchimento labial, se emocionou ao mostrar a novidade e contou que tem esse sonho desde a adolescência:
--------------------------------------------------
Novos resultados da avaliação Rouge:

--- ROUGE1 ---
Precision: 30.00%
Recall   : 46.88%
F1-Score : 36.59%

--- ROUGE2 ---
Precision: 8.16%
Recall   : 12.90%
F1-Score : 10.00%

--- ROUGEL ---
Precision: 14.00%
Recall   : 21.88%
F1-Score : 17.07%



In [19]:
# TESTE DE MUDANÇA DE FRASES SELECIONADAS NO ALGORITMO TF-IDF (BERTSCORE)

import evaluate

# Recalcular o resumo com 2 frases para garantir a consistência das variáveis
qtdd_frases_resumo = 2
melhores_frases = frases_ord[:qtdd_frases_resumo]
melhores_frases_cron = sorted(melhores_frases, key=lambda x: x[0])
resumo_gerado_2_frases = " ".join([frase[2] for frase in melhores_frases_cron])

# Inicializar o avaliador semântico
avaliador_bertscore = evaluate.load("bertscore")

# Calcular a pontuação semântica
pontuacao_bert_2_frases = avaliador_bertscore.compute(
    predictions=[resumo_gerado_2_frases],
    references=[resumo_ref],
    lang="pt"
)

# Exibição dos resultados
print("Resumo gerado (2 Frases):\n", resumo_gerado_2_frases)
print("-" * 50)
print("Novos resultados da avaliação BERTScore:\n")
print("--- BERTSCORE (Análise Semântica) ---")
print(f"Precision: {pontuacao_bert_2_frases['precision'][0] * 100:.2f}%")
print(f"Recall   : {pontuacao_bert_2_frases['recall'][0] * 100:.2f}%")
print(f"F1-Score : {pontuacao_bert_2_frases['f1'][0] * 100:.2f}%\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Resumo gerado (2 Frases):
 A Yalochat, startup que oferece inteligência artificial para atendimento ao cliente via WhatsApp, coleta de dados e outras soluções de vendas online, vai receber US$ 15 milhões (ou R$ 82 milhões) do fundo B Capital Group numa segunda rodada de investimentos. Pagamento instantâneo PIX vai permitir saque direto no varejo, diz Campos NetoCom função de pagamentos, WhatsApp pode se tornar ‘super app’ e ameaçar bancosPIX, do Banco Central, pode acelerar fim do papel moeda e reduzir informalidade Mesmo as grandes companhias, como essas, precisaram se adaptar (mais rápido) para manter as portas abertas.
--------------------------------------------------
Novos resultados da avaliação BERTScore:

--- BERTSCORE (Análise Semântica) ---
Precision: 65.98%
Recall   : 73.39%
F1-Score : 69.49%



In [20]:
# TESTE DEFINITIVO EM LOTES (ROUGE + BERTSCORE)

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from rouge_score import rouge_scorer
import evaluate

qtdd_ntc = 100
qtdd_frases_resumo = 2

# Instanciando os avaliadores ANTES do laço
avaliador_rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
print("Carregando o modelo semântico do BERTScore...")
avaliador_bertscore = evaluate.load("bertscore")

# Dicionário para guardar as notas do ROUGE
notas_f1 = {'rouge1': [], 'rouge2': [], 'rougeL': []}

# Listas para guardar os textos para o BERTScore avaliar em lote no final
lista_resumos_gerados = []
lista_resumos_referencia = []

print(f"Iniciando o processamento em lote de {qtdd_ntc} notícias...\n")

# laço de repetição para percorrer as notícias
for i in range(qtdd_ntc):
    ntc_orig = table_treino['Noticia'][i]
    resumo_ref = table_treino['Sumario'][i]

    # validação 1: pular o texto se for muito curto ou inválido
    if not isinstance(ntc_orig, str) or len(ntc_orig) < 50:
        continue

    # processamento
    doc_spacy = nlp(ntc_orig)
    frases_orig = [sentenca.text for sentenca in doc_spacy.sents]

    # validação 2: pular se a notícia tiver menos frases que o limite de resumo
    if len(frases_orig) <= qtdd_frases_resumo:
        continue

    frases_limp = [pre_proc_text(frase) for frase in frases_orig]

    # tf-idf
    try:
        vetorizador = TfidfVectorizer()
        matriz_tfidf = vetorizador.fit_transform(frases_limp)
    except ValueError:  # verificação 3: evitar erro caso alguma notícia fique completamente vazia
        continue

    pontuacoes = np.array(matriz_tfidf.sum(axis=1)).flatten()

    # sumarização
    frases_com_pont = [(indice, pont, frases_orig[indice]) for indice, pont in enumerate(pontuacoes)]
    frases_ord = sorted(frases_com_pont, key=lambda x: x[1], reverse=True)

    melhores_frases_cron = sorted(frases_ord[:qtdd_frases_resumo], key=lambda x: x[0])
    resumo_gerado = " ".join([frase[2] for frase in melhores_frases_cron])

    # --- AVALIAÇÃO ROUGE (Individual) ---
    pont_rouge = avaliador_rouge.score(resumo_ref, resumo_gerado)
    notas_f1['rouge1'].append(pont_rouge['rouge1'].fmeasure)
    notas_f1['rouge2'].append(pont_rouge['rouge2'].fmeasure)
    notas_f1['rougeL'].append(pont_rouge['rougeL'].fmeasure)

    # --- PREPARAÇÃO BERTSCORE ---
    # Guardamos os textos para avaliar todos de uma vez no final
    lista_resumos_gerados.append(resumo_gerado)
    lista_resumos_referencia.append(resumo_ref)

    # print do progresso a cada 10 notícias
    if (i+1) % 10 == 0:
        print(f"Notícias processadas: {i + 1}/{qtdd_ntc}...")

# --- AVALIAÇÃO BERTSCORE (Em Lote) ---
print("\nProcessamento TF-IDF concluído! Calculando as notas semânticas (BERTScore)...")
resultados_bertscore = avaliador_bertscore.compute(
    predictions=lista_resumos_gerados,
    references=lista_resumos_referencia,
    lang="pt"
)

# cálculo das médias finais
media_r1 = np.mean(notas_f1['rouge1']) * 100
media_r2 = np.mean(notas_f1['rouge2']) * 100
media_rl = np.mean(notas_f1['rougeL']) * 100
media_bert = np.mean(resultados_bertscore['f1']) * 100

# exibição de resultados
print("\n" + "=" * 55)
print(f"Média de desempenho do modelo ({len(lista_resumos_gerados)} notícias válidas)")
print("Métrica Principal: Medida F1 (F1-Score)")
print("=" * 55)
print(f"Rouge-1 (Palavras exatas):   {media_r1:.2f}%")
print(f"Rouge-2 (Frases exatas):     {media_r2:.2f}%")
print(f"Rouge-L (Sequência exata):   {media_rl:.2f}%")
print(f"BERTScore (Sentido/Contexto):{media_bert:.2f}%")
print("=" * 55)

Carregando o modelo semântico do BERTScore...
Iniciando o processamento em lote de 100 notícias...

Notícias processadas: 10/100...
Notícias processadas: 20/100...
Notícias processadas: 30/100...
Notícias processadas: 40/100...
Notícias processadas: 50/100...
Notícias processadas: 60/100...
Notícias processadas: 70/100...
Notícias processadas: 80/100...
Notícias processadas: 90/100...
Notícias processadas: 100/100...

Processamento TF-IDF concluído! Calculando as notas semânticas (BERTScore)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Média de desempenho do modelo (97 notícias válidas)
Métrica Principal: Medida F1 (F1-Score)
Rouge-1 (Palavras exatas):   27.86%
Rouge-2 (Frases exatas):     10.39%
Rouge-L (Sequência exata):   18.30%
BERTScore (Sentido/Contexto):70.09%
